# JAX Runner: Efficient Local Attention + Span-Hypergraph + Compressed Memory LM

Trains the pure-JAX implementation in `jax_model/` on a Hugging Face **streaming**
dataset (same pipeline as the original torch notebook: documents are tokenized on the
fly and packed into contiguous `block_size + 1` token chunks — nothing is downloaded
up front).

Runs on a single device (CPU / Apple Metal / one CUDA GPU) **or data-parallel on any
multi-device setup (TPU slice, multi-GPU node)**: when more than one device is
visible, parameters are replicated, each batch is sharded across devices along the
batch axis, and XLA's GSPMD inserts the gradient all-reduce automatically — the train
step code is identical either way.

Notes:

- Parameters are initialized by building the torch model once and converting with
  `jax_model.from_torch_model` (the repo's tested parity path); torch is not used after that.
- The optimizer is a self-contained AdamW + cosine schedule + global-norm clipping in
  pure JAX (no extra dependencies on top of the repo's pinned `jax==0.4.34` / jax-metal stack).
- `attention_backend="chunked"` is O(T·window) on every backend and is the only
  attention backend whose backward pass compiles on Apple Metal.
- On TPU, compute defaults to **bfloat16** (master weights, optimizer state, and the
  cross-entropy stay in fp32); elsewhere everything is fp32 — flip `use_bfloat16` on
  manually for Ampere+ CUDA GPUs. The JAX forward has no dropout (deterministic).

## 0. TPU / CUDA setup (skip on a local machine)

On a fresh Cloud TPU VM or Colab/Kaggle TPU runtime, uncomment the TPU line; on a
CUDA machine use the `jax[cuda12]` line instead (the repo's pinned `jax==0.4.34` +
`jax-metal` is for Apple Silicon only). Run once, then restart the kernel. Colab TPU
runtimes usually ship with a TPU-enabled `jax` already, in which case only the
clone + datasets/transformers/torch lines are needed. Torch is CPU-only here — it is
used solely for parameter initialization.

In [ ]:
# !git clone https://github.com/HyperRays/Hyperattn
# %cd Hyperattn
# !pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html   # TPU
# !pip install -U numpy "jax[cuda12]"   # NVIDIA GPU (recent jax needs numpy>=2.0 — restart kernel after)
# !pip install datasets transformers
# !pip install torch --index-url https://download.pytorch.org/whl/cpu

## 1. Imports and configuration

In [1]:
import math
import pickle
import time
from dataclasses import asdict
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np

import jax_model

n_devices = jax.device_count()
print("jax backend:", jax.default_backend())
print("jax devices:", n_devices, jax.devices())

seed = 1337

# -----------------------
# User-editable settings
# -----------------------

# Dataset: FineWeb-Edu sample-10BT, streamed. Swap to e.g. "roneneldan/TinyStories"
# (dataset_config=None) for a smaller/debug dataset.
dataset_name = "HuggingFaceFW/fineweb-edu"
dataset_config = "sample-10BT"
dataset_split = "train"
text_field = "text"
tokenizer_name = "gpt2"
shuffle_buffer = 50_000
val_docs = 2_000  # held out by taking the first docs from the stream

# Training shape. Per-device batch of 4; the global batch scales with device count
# (4 on a laptop or single GPU, 32 on a TPU v3-8 or an 8-GPU node). Must stay
# divisible by n_devices.
batch_size = 4 * n_devices
block_size = 1024
assert batch_size % n_devices == 0

# Compute dtype: bf16 on TPU (weights/optimizer/cross-entropy stay fp32), fp32
# elsewhere. On an Ampere-or-newer NVIDIA GPU, set use_bfloat16 = True manually.
use_bfloat16 = jax.default_backend() == "tpu"
compute_dtype = jnp.bfloat16 if use_bfloat16 else jnp.float32
print("global batch:", batch_size, "| compute dtype:", compute_dtype.__name__)

# Model size (same defaults as the torch notebook).
n_embd = 384
n_head = 6
n_local_attn_layers = 1
n_span_layers = 6
n_compressed_memory_layers = 1
span_widths = (2, 4, 8, 16, 32, 64)
local_window = 256
compression_block = 64

# Block stack (overrides the n_*_layers counts above). Deep ~100M layout: one local
# attention, then sandwiches of [4 span, HCA, 4 span] so HCA sits mid-stack with span
# layers above and below it to consume its global signal.
block_layout = tuple(["attn"] + (["span"] * 4 + ["hca"] + ["span"] * 4) * 3 + ["span"] * 4 + ["hca"] + ["span"] * 2)
ablation_interval = 2500  # log per-block importance via ablation this often (0 disables)

# Optimization.
max_iters = 10_000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-4
min_lr_ratio = 0.1
warmup_iters = 200
weight_decay = 0.1
grad_clip = 1.0

# Optimizer: Muon (Newton-Schulz-orthogonalized momentum) on the 2D weight matrices,
# AdamW on everything else (token embedding, norm gains, biases, gates). muon_rms_scale
# matches Muon's update RMS to AdamW's so the same lr schedule transfers.
use_muon = True
muon_momentum = 0.95
muon_ns_steps = 5
muon_rms_scale = 0.2
muon_lr_mult = 1.0
save_best_checkpoint = True
checkpoint_path = "best_jax_span_hypergraph_lm.pkl"

# JAX backends (see README): chunked attention trains everywhere and scales linearly.
attention_backend = "chunked"
span_backend = "materialized"

# Generation.
generate_tokens = 200
temperature = 0.8
top_k = 50
prompt = "The meaning of intelligence is"

Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!


Metal device set to: Apple M3

systemMemory: 16.00 GB
maxCacheSize: 5.92 GB

jax backend: METAL
jax devices: 1 [METAL(id=0)]
global batch: 4 | compute dtype: float32


W0000 00:00:1781348064.974001 4580674 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1781348064.990217 4580674 service.cc:145] XLA service 0xae8d77f00 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781348064.990400 4580674 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1781348064.991787 4580674 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1781348064.991795 4580674 mps_client.cc:384] XLA backend will use up to 12712640512 bytes on device 0 for SimpleAllocator.


## 2. Streaming packed-token dataset

Adapted from the original notebook's `StreamingPackedTokenDataset`, minus the torch
`DataLoader`: a plain generator that yields `(x, y)` NumPy int32 batches of shape
`(batch_size, block_size)`. The first `val_docs` documents are held out for validation;
the training stream skips them and shuffles with a buffer.

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer


class StreamingPackedTokens:
    def __init__(self, shuffle, seed, skip_docs=0, take_docs=None):
        self.shuffle = shuffle
        self.seed = seed
        self.skip_docs = skip_docs
        self.take_docs = take_docs
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
        if self.tokenizer.eos_token_id is None:
            self.tokenizer.add_special_tokens({"eos_token": "<|endoftext|>"})
        self.eos_id = self.tokenizer.eos_token_id

    def _make_stream(self):
        if dataset_config is None:
            ds = load_dataset(dataset_name, split=dataset_split, streaming=True)
        else:
            ds = load_dataset(dataset_name, dataset_config, split=dataset_split, streaming=True)
        # Validation stream should be deterministic; training stream should be shuffled.
        if self.shuffle:
            ds = ds.shuffle(buffer_size=shuffle_buffer, seed=self.seed)
        if self.skip_docs:
            ds = ds.skip(self.skip_docs)
        if self.take_docs is not None:
            ds = ds.take(self.take_docs)
        return ds

    def _chunks(self):
        # Infinite stream for training; one finite pass when take_docs is set.
        while True:
            token_buffer = []
            yielded_any = False
            for row in self._make_stream():
                text = row.get(text_field, None)
                if not isinstance(text, str) or len(text) == 0:
                    continue
                ids = self.tokenizer.encode(text, add_special_tokens=False)
                ids.append(self.eos_id)
                token_buffer.extend(ids)
                while len(token_buffer) >= block_size + 1:
                    chunk = np.asarray(token_buffer[: block_size + 1], dtype=np.int32)
                    token_buffer = token_buffer[block_size + 1 :]
                    yielded_any = True
                    yield chunk[:-1], chunk[1:]
            if self.take_docs is not None:
                break
            if not yielded_any:
                raise RuntimeError("Dataset iterator yielded no examples. Check dataset config/text field.")

    def batches(self):
        xs, ys = [], []
        for x, y in self._chunks():
            xs.append(x)
            ys.append(y)
            if len(xs) == batch_size:
                yield np.stack(xs), np.stack(ys)
                xs, ys = [], []


train_ds = StreamingPackedTokens(shuffle=True, seed=seed, skip_docs=val_docs)
val_ds = StreamingPackedTokens(shuffle=False, seed=seed, take_docs=val_docs)
tokenizer = train_ds.tokenizer
vocab_size = len(tokenizer)
print("vocab_size:", vocab_size)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)


def cycle_batches(ds):
    while True:
        yield from ds.batches()


train_iter = cycle_batches(train_ds)
val_iter = cycle_batches(val_ds)

/Users/soham/Documents/programming/Hyperattn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab_size: 50257
eos token: <|endoftext|> 50256


## 3. Model initialization

Build the torch model once with the repo's tested initialization, convert to a JAX
parameter pytree, then drop the torch model. `dropout=0.0` because the JAX forward
pass is deterministic.

In [4]:
import torch

from model import EfficientHGConfig, EfficientHypergraphLM

torch.manual_seed(seed)
torch_cfg = EfficientHGConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_local_attn_layers=n_local_attn_layers,
    n_span_layers=n_span_layers,
    n_compressed_memory_layers=n_compressed_memory_layers,
    span_widths=span_widths,
    local_window=local_window,
    compression_block=compression_block,
    dropout=0.0,
    block_layout=block_layout,
)
params, cfg = jax_model.from_torch_model(EfficientHypergraphLM(torch_cfg))
print(f"parameters: {jax_model.count_parameters(params) / 1e6:.2f}M  ({len(block_layout)} blocks)")
print("layout:", block_layout)

parameters: 37.03M
config: {'vocab_size': 50257, 'block_size': 1024, 'n_embd': 384, 'n_head': 6, 'n_local_attn_layers': 1, 'n_span_layers': 6, 'n_compressed_memory_layers': 1, 'span_widths': (2, 4, 8, 16, 32, 64), 'local_window': 256, 'compression_block': 64, 'dropout': 0.0}


## 4. Device mesh and sharding

Standard data parallelism: a 1-D `"data"` mesh, parameters/optimizer state replicated,
batches sharded along the batch axis. With sharded inputs, `jax.jit` (GSPMD) partitions
the computation and all-reduces gradients without any change to the step function.
On a single device both helpers are no-ops.

In [9]:
if n_devices > 1:
    from jax.sharding import Mesh, NamedSharding, PartitionSpec

    mesh = Mesh(np.asarray(jax.devices()), ("data",))
    _replicated = NamedSharding(mesh, PartitionSpec())
    _data_sharded = NamedSharding(mesh, PartitionSpec("data"))

    def replicate(tree):
        return jax.device_put(tree, _replicated)

    def shard_batch(a):
        return jax.device_put(jnp.asarray(a), _data_sharded)
else:
    def replicate(tree):
        return tree

    shard_batch = jnp.asarray

params = replicate(params)

## 5. Optimizer and train/eval steps

Self-contained AdamW (decoupled weight decay on all parameters, like the torch
notebook's `torch.optim.AdamW(model.parameters(), ...)`), cosine LR schedule with
warmup, and global-norm gradient clipping. The whole update is one jitted step with
donated params/optimizer buffers; `lr` is a traced argument so the schedule does not
trigger recompiles. When `use_bfloat16` is on, weights are cast to bf16 inside the
loss (gradients and the AdamW update stay fp32) and the logits are cast back to fp32
before the cross-entropy.

In [8]:
from jax_model.ops import softmax_cross_entropy


def get_lr(step):
    if step < warmup_iters:
        return learning_rate * step / max(1, warmup_iters)
    if step > max_iters:
        return learning_rate * min_lr_ratio
    decay_ratio = (step - warmup_iters) / max(1, max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * min_lr_ratio + coeff * (learning_rate - learning_rate * min_lr_ratio)


def _is_muon_leaf(path, x):
    # Muon only on real 2D weight matrices; embedding / norms / biases / gates -> AdamW.
    if x.ndim != 2 or min(x.shape) == 1:
        return False
    return not any("token_embedding" in str(getattr(k, "key", k)) for k in path)


def newton_schulz(G, steps):
    # quintic Newton-Schulz: orthogonalize G (singular values -> 1), in bf16
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.astype(jnp.bfloat16)
    X = X / (jnp.linalg.norm(X) + 1e-7)
    transpose = G.shape[0] > G.shape[1]
    if transpose:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * (A @ A)) @ X
    if transpose:
        X = X.T
    return X.astype(G.dtype)


muon_mask = jax.tree_util.tree_map_with_path(_is_muon_leaf, params)
print(f"optimizer: Muon on {sum(1 for m in jax.tree.leaves(muon_mask) if m)} 2D matrices + AdamW on the rest"
      if use_muon else "optimizer: AdamW (use_muon=False)")


def opt_init(params):
    zeros = lambda: jax.tree.map(jnp.zeros_like, params)
    return {"s1": zeros(), "s2": zeros(), "step": jnp.zeros((), dtype=jnp.int32)}


def opt_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    grad_norm = jnp.sqrt(sum(jnp.sum(jnp.square(g)) for g in jax.tree.leaves(grads)))
    finite = jnp.isfinite(grad_norm)
    clip_scale = jnp.minimum(1.0, grad_clip / (grad_norm + 1e-12))
    grads = jax.tree.map(lambda g: g * clip_scale, grads)

    step = state["step"] + 1
    bc1 = 1 - beta1 ** step.astype(jnp.float32)
    bc2 = 1 - beta2 ** step.astype(jnp.float32)
    treedef = jax.tree_util.tree_structure(params)
    pl, gl, s1l, s2l, ml = (jax.tree.leaves(t) for t in (params, grads, state["s1"], state["s2"], muon_mask))
    new_p, new_s1, new_s2 = [], [], []
    for p, g, s1, s2, use in zip(pl, gl, s1l, s2l, ml):
        if use_muon and use:  # Muon: Nesterov momentum -> Newton-Schulz orthogonalization
            buf = muon_momentum * s1 + g
            ortho = newton_schulz(g + muon_momentum * buf, muon_ns_steps)
            upd = (muon_lr_mult * muon_rms_scale * math.sqrt(max(p.shape))) * ortho
            new_p.append(p - lr * (upd + weight_decay * p))
            new_s1.append(buf)
            new_s2.append(s2)
        else:  # AdamW
            m = beta1 * s1 + (1 - beta1) * g
            v = beta2 * s2 + (1 - beta2) * jnp.square(g)
            new_p.append(p - lr * ((m / bc1) / (jnp.sqrt(v / bc2) + eps) + weight_decay * p))
            new_s1.append(m)
            new_s2.append(v)

    update_norm = jnp.sqrt(sum(jnp.sum(jnp.square(a - b)) for a, b in zip(new_p, pl)))
    param_norm = jnp.sqrt(sum(jnp.sum(jnp.square(pp)) for pp in pl))
    update_ratio = update_norm / (param_norm + 1e-12)

    keep = lambda new, old: jnp.where(finite, new, old)
    new_params = jax.tree.map(keep, jax.tree_util.tree_unflatten(treedef, new_p), params)
    new_state = {
        "s1": jax.tree.map(keep, jax.tree_util.tree_unflatten(treedef, new_s1), state["s1"]),
        "s2": jax.tree.map(keep, jax.tree_util.tree_unflatten(treedef, new_s2), state["s2"]),
        "step": keep(step, state["step"]),
    }
    return new_params, new_state, grad_norm, update_ratio


def loss_fn(p, x, y):
    if compute_dtype != jnp.float32:
        p = jax.tree.map(
            lambda a: a.astype(compute_dtype) if jnp.issubdtype(a.dtype, jnp.floating) else a, p
        )
    logits = jax_model.forward(p, x, cfg, attention_backend=attention_backend, span_backend=span_backend)
    return softmax_cross_entropy(logits.astype(jnp.float32), y)


@partial(jax.jit, donate_argnums=(0, 1))
def train_step(params, opt_state, x, y, lr):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    params, opt_state, grad_norm, update_ratio = opt_update(params, grads, opt_state, lr)
    return params, opt_state, loss, grad_norm, update_ratio


eval_step = jax.jit(loss_fn)


def estimate_loss():
    losses = []
    for _ in range(eval_iters):
        xb, yb = next(val_iter)
        losses.append(eval_step(params, shard_batch(xb), shard_batch(yb)))
    return float(jnp.mean(jnp.stack(losses)))


def print_gates(params):
    sig = lambda r: 1 / (1 + math.exp(-r))
    span_g, hca_g = [], []
    for i, block in enumerate(params["blocks"]):
        if "gate" not in block:
            continue
        (hca_g if "pool_score" in block else span_g).append((i, sig(float(block["gate"]))))
    if span_g:
        v = [s for _, s in span_g]
        print(f"  span gates ({len(v)}): min {min(v):.3f}  mean {sum(v)/len(v):.3f}  max {max(v):.3f}")
    if hca_g:
        print("  hca gates: " + "  ".join(f"b{i}:{s:.3f}" for i, s in hca_g))


opt_state = replicate(opt_init(params))

NameError: name 'replicate' is not defined

## 6. Forward/backward smoke test

The first call compiles (slow); the second shows the steady-state step time.
(On Apple Metal a "donation is not implemented" warning is expected and harmless.)

In [7]:
xb, yb = next(train_iter)
xb, yb = shard_batch(xb), shard_batch(yb)
print("x:", xb.shape, "y:", yb.shape)

t0 = time.time()
params, opt_state, loss, grad_norm, update_ratio = train_step(params, opt_state, xb, yb, get_lr(1))
loss.block_until_ready()
print(f"first step (incl. compile): {time.time() - t0:.1f}s  loss={float(loss):.4f}")

xb, yb = next(train_iter)
xb, yb = shard_batch(xb), shard_batch(yb)
t0 = time.time()
params, opt_state, loss, grad_norm, update_ratio = train_step(params, opt_state, xb, yb, get_lr(2))
loss.block_until_ready()
dt = time.time() - t0
print(f"steady-state step: {dt * 1000:.0f}ms  ({xb.size / dt:,.0f} tok/s)  "
      f"loss={float(loss):.4f}  grad_norm={float(grad_norm):.3f}  update/param={float(update_ratio):.2e}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1165 > 1024). Running this sequence through the model will result in indexing errors


x: (4, 1024) y: (4, 1024)


/Users/soham/Documents/programming/Hyperattn/.venv/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1101: UserWarning: Some donated buffers were not usable: ShapedArray(float32[384,384]), ShapedArray(float32[1152,384]), ShapedArray(float32[384]), ShapedArray(float32[384]), ShapedArray(float32[384]), ShapedArray(float32[384]), ShapedArray(float32[1536]), ShapedArray(float32[1536,384]), ShapedArray(float32[384]), ShapedArray(float32[384,1536]), ShapedArray(float32[384]), ShapedArray(float32[384,2304]), ShapedArray(float32[]), ShapedArray(float32[384]), ShapedArray(float32[384,384]), ShapedArray(float32[384]), ShapedArray(float32[384]), ShapedArray(float32[384]), ShapedArray(float32[384]), ShapedArray(float32[1536]), ShapedArray(float32[1536,384]), ShapedArray(float32[384]), ShapedArray(float32[384,1536]), ShapedArray(float32[384]), ShapedArray(float32[384,384]), ShapedArray(float32[384]), ShapedArray(float32[384,2304]), ShapedArray(float32[]), ShapedArray(float32[384]), ShapedA

first step (incl. compile): 11.3s  loss=10.8827
steady-state step: 2110ms  (1,941 tok/s)  loss=10.8573  grad_norm=2.515


## 7. Resume (optional)

For long runs (e.g. 1M steps), the training loop writes a resumable `latest`
checkpoint every `eval_interval` — full optimizer moments + step + bookkeeping, not
just weights. To continue after a crash, set `resume = True` here and re-run the
notebook from the top. Leave it `False` to start fresh.

In [ ]:
# Resume support. The training loop writes a resumable "latest" checkpoint every
# eval_interval (params + optimizer moments + step + bookkeeping), so a crashed long
# run can pick up near where it stopped. To continue a run, set resume = True and
# re-run the notebook top to bottom: this cell restores the full optimizer state so
# Adam's moments and the LR schedule continue (a params-only reload would restart
# Adam cold at step 0). On resume it overwrites the fresh params/opt_state that the
# cells above just built, so those cells are safe to run.
resume = False
latest_checkpoint_path = "latest_jax_span_hypergraph_lm.pkl"


def save_checkpoint(path, step, val_loss):
    payload = {
        "params": jax.tree.map(np.asarray, jax.device_get(params)),
        "opt_state": jax.tree.map(np.asarray, jax.device_get(opt_state)),
        "config": asdict(cfg),
        "step": step,
        "best_val": best_val,
        "val_loss": val_loss,
        "train_loss_ema": loss_ema,
        "total_tokens": total_tokens,
        "tokenizer_name": tokenizer_name,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f)


start_step = 0
best_val = float("inf")
loss_ema = None
total_tokens = 0

if resume:
    with open(latest_checkpoint_path, "rb") as f:
        ckpt = pickle.load(f)
    params = replicate(jax.tree.map(jnp.asarray, ckpt["params"]))
    opt_state = replicate(jax.tree.map(jnp.asarray, ckpt["opt_state"]))
    start_step = ckpt["step"] + 1
    best_val = ckpt.get("best_val", ckpt.get("val_loss", float("inf")))
    loss_ema = ckpt["train_loss_ema"]
    total_tokens = ckpt["total_tokens"]
    # Reshuffle the training stream from a step-derived seed so we do not replay the
    # same opening documents (HF streaming cannot checkpoint an exact position).
    train_ds = StreamingPackedTokens(shuffle=True, seed=seed + start_step, skip_docs=val_docs)
    train_iter = cycle_batches(train_ds)
    print(f"resumed: step {start_step}, best_val {best_val:.4f}, tokens {total_tokens:,}")
else:
    print("fresh start (resume=False)")

## 8. Training loop

In [ ]:
# best_val / loss_ema / total_tokens / start_step come from the resume cell above.
tokens_since_eval = 0
steps_since_eval = 0
clip_steps = 0
nonfinite_steps = 0
gnorm_ema = None
uratio_ema = None
t0 = time.time()
abl_xb, abl_yb = next(val_iter)  # fixed batch -> comparable diagnostics across time

for step in range(start_step, max_iters + 1):
    if step % eval_interval == 0 or step == max_iters:
        elapsed = time.time() - t0
        toks_per_sec = 0.0 if step == start_step else tokens_since_eval / max(elapsed, 1e-9)
        val_loss = estimate_loss()
        train_loss_str = "nan" if loss_ema is None else f"{loss_ema:.4f}"
        print(
            f"step {step:6d} | train_ema {train_loss_str} | val {val_loss:.4f} | "
            f"lr {get_lr(step):.2e} | tok/s {toks_per_sec:,.0f} | "
            f"tokens {total_tokens:,} | elapsed {elapsed:.1f}s"
        )
        print_gates(params)
        clip_frac = clip_steps / max(1, steps_since_eval)
        gn_str = "nan" if gnorm_ema is None else f"{gnorm_ema:.3f}"
        ur_str = "nan" if uratio_ema is None else f"{uratio_ema:.2e}"
        print(f"  grad_norm_ema {gn_str} | update/param {ur_str} | clipped {clip_frac:.0%} | nonfinite {nonfinite_steps}")
        if ablation_interval and step % ablation_interval == 0:
            _, deltas = jax_model.block_ablation_deltas(
                params, shard_batch(abl_xb), shard_batch(abl_yb), cfg,
                attention_backend=attention_backend, span_backend=span_backend,
            )
            dead = sum(1 for _, _, d in deltas if abs(d) < 0.01)
            print(f"  block importance (Δloss when mixer ablated)  [near-dead <0.01: {dead}/{len(deltas)}]:")
            print("    " + "  ".join(f"{i}:{k[0]}{d:+.3f}" for i, k, d in deltas))
            diag = jax_model.backbone_diagnostics(
                params, shard_batch(abl_xb), cfg,
                attention_backend=attention_backend, span_backend=span_backend,
            )
            rms = diag["residual_rms"]
            print(f"  residual RMS: {rms[0]:.2f} -> {rms[-1]:.2f} (max {max(rms):.2f})")
            if diag["hca_null_mass"]:
                print("  hca null-sink mass: "
                      + "  ".join(f"b{i}:{m:.3f}" for i, m in diag["hca_null_mass"].items())
                      + "   (->1 = memory bypassed)")
            pos = jax_model.position_bucketed_loss(
                params, shard_batch(abl_xb), shard_batch(abl_yb), cfg,
                attention_backend=attention_backend, span_backend=span_backend,
            )
            print("  loss by position: " + "  ".join(f"[{lo}-{hi}]{l:.3f}" for lo, hi, l in pos))
        t0 = time.time()
        tokens_since_eval = 0
        steps_since_eval = 0
        clip_steps = 0
        nonfinite_steps = 0

        # Always refresh the resumable "latest" checkpoint; keep the best separately.
        save_checkpoint(latest_checkpoint_path, step, val_loss)
        if save_best_checkpoint and val_loss < best_val:
            best_val = val_loss
            save_checkpoint(checkpoint_path, step, val_loss)
            print(f"saved best checkpoint to {checkpoint_path} with val={best_val:.4f}")

    if step == max_iters:
        break

    xb, yb = next(train_iter)
    params, opt_state, loss, grad_norm, update_ratio = train_step(
        params, opt_state, shard_batch(xb), shard_batch(yb), get_lr(step)
    )

    # one host sync per step for all scalar metrics (cheaper on TPU than separate pulls)
    loss_value, gnorm, uratio = (float(v) for v in np.asarray(jnp.stack([loss, grad_norm, update_ratio])))
    if not (math.isfinite(loss_value) and math.isfinite(gnorm)):
        nonfinite_steps += 1  # the optimizer step was already skipped on-device; keep training
        continue
    loss_ema = loss_value if loss_ema is None else 0.99 * loss_ema + 0.01 * loss_value
    gnorm_ema = gnorm if gnorm_ema is None else 0.99 * gnorm_ema + 0.01 * gnorm
    uratio_ema = uratio if uratio_ema is None else 0.99 * uratio_ema + 0.01 * uratio
    clip_steps += int(gnorm > grad_clip)
    steps_since_eval += 1
    tokens_since_eval += xb.size
    total_tokens += xb.size

## 9. Generate text

All blocks are causal, so generation runs the model over a fixed-size buffer (one jit
compile) and reads the logits at the current position; the buffer slides once full.
Generation is fp32 and batch-1; on a multi-device mesh it simply runs replicated.

In [13]:
# Optionally load a checkpoint before generation (best by val, or the most recent):
with open(checkpoint_path, "rb") as f:          # or latest_checkpoint_path
    ckpt = pickle.load(f)
params = replicate(jax.tree.map(jnp.asarray, ckpt["params"]))

gen_window = min(block_size, 512)


@jax.jit
def gen_logits(p, idx):
    return jax_model.forward(p, idx, cfg, attention_backend=attention_backend, span_backend=span_backend)


def generate(params, prompt, max_new_tokens, temperature=0.8, top_k=50,
             rep_penalty=1.2, rep_window=128, seed=0):
    rng = np.random.default_rng(seed)
    ids = tokenizer.encode(prompt)
    out = list(ids)
    ids = ids[-gen_window:]
    buf = np.full((1, gen_window), tokenizer.eos_token_id, dtype=np.int32)
    buf[0, : len(ids)] = ids
    cur = len(ids)
    for _ in range(max_new_tokens):
        logits = np.asarray(gen_logits(params, jnp.asarray(buf)))[0, cur - 1].astype(np.float64)
        recent = np.fromiter(set(out[-rep_window:]), dtype=np.int64)   # penalize recent tokens
        pos = logits[recent] > 0
        logits[recent] = np.where(pos, logits[recent] / rep_penalty, logits[recent] * rep_penalty)
        logits = logits / temperature
        if top_k is not None:
            cutoff = np.partition(logits, -top_k)[-top_k]
            logits = np.where(logits < cutoff, -np.inf, logits)
        probs = np.exp(logits - logits.max()); probs /= probs.sum()
        nxt = int(rng.choice(len(probs), p=probs))
        out.append(nxt)
        if cur == gen_window:
            buf[0, :-1] = buf[0, 1:]; cur -= 1
        buf[0, cur] = nxt; cur += 1
    return tokenizer.decode(out)


print(generate(params, prompt, generate_tokens, temperature=0.85, top_k=50, seed=seed))

The meaning of intelligence is its ability to distinguish itself from the information it receives, and thus it is not as a reward-service.
To ensure that the message conveyed through the information reaches out across your home, you need to understand what it takes to know all these details so far. It is this knowledge that in addition to personal information which is sent by the information and information that they are communicating with, your ability to remember should be strictly understood. In fact, information may not always be found at the outset when you are discussing what one would think of for you or your friends. A little bit more than just describing things; there needs to be an understanding of how words can relate and do a better job that's going on.<|endoftext|>The United States Department of Agriculture has reported their first-ever average yearly water consumption rate in 2009; 2 deaths for livestock feeders per 1-1 year at sea level
In a paper published online Januar